In [1]:
"""
Credit Card Fraud Detection - EDA, Modeling & Explainability
================================================================
Dataset: Kaggle "Credit Card Fraud Detection" (mlg-ulb/creditcardfraud)
Download: https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud
Place the file as: data/creditcard.csv

284,807 transactions, 492 frauds (0.17%) — real anonymized European
cardholder transactions. Features V1-V28 are PCA components (already
anonymized); Time and Amount are raw.

Run this as a script or paste cells into a Jupyter notebook.
"""

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, average_precision_score, RocCurveDisplay
)
from imblearn.over_sampling import SMOTE
import xgboost as xgb
import shap
import joblib

RANDOM_STATE = 42

# -----------------------------------------------------------------
# 1. LOAD DATA
# -----------------------------------------------------------------
df = pd.read_csv(r"C:\Users\rkdas\OneDrive\Documents\Desktop\credit_card_fraud_detection\creditcard.csv\creditcard.csv")
print(df.shape)
print(df["Class"].value_counts(normalize=True))  # ~0.17% fraud -> extreme imbalance

(284807, 31)
Class
0    0.998273
1    0.001727
Name: proportion, dtype: float64


In [2]:
# -----------------------------------------------------------------
# 2. EDA
# -----------------------------------------------------------------
# Class imbalance plot
sns.countplot(x="Class", data=df)
plt.title("Class distribution (0 = legit, 1 = fraud)")
plt.savefig("outputs/class_distribution.png", bbox_inches="tight")
plt.close()

In [3]:
# Amount distribution: fraud vs legit
plt.figure(figsize=(8, 4))
sns.kdeplot(df[df.Class == 0]["Amount"], label="Legit", clip=(0, 500))
sns.kdeplot(df[df.Class == 1]["Amount"], label="Fraud", clip=(0, 500))
plt.title("Transaction amount distribution by class")
plt.legend()
plt.savefig("outputs/amount_distribution.png", bbox_inches="tight")
plt.close()

In [4]:
# Correlation of top features with target (PCA components are already
# decorrelated by construction, so this mostly confirms which V-features
# separate classes best)
corrs = df.corr()["Class"].abs().sort_values(ascending=False)[1:11]
print("Top 10 features correlated with fraud:\n", corrs)

Top 10 features correlated with fraud:
 V17    0.326481
V14    0.302544
V12    0.260593
V10    0.216883
V16    0.196539
V3     0.192961
V7     0.187257
V11    0.154876
V4     0.133447
V18    0.111485
Name: Class, dtype: float64


In [5]:
# WHY: Time and Amount are on totally different scales from the PCA
# components (V1-V28 are already standardized). Tree models don't care,
# but Logistic Regression does -> scale them.
scaler = StandardScaler()
df["Amount_scaled"] = scaler.fit_transform(df[["Amount"]])
df["Time_scaled"] = scaler.fit_transform(df[["Time"]])
df = df.drop(["Amount", "Time"], axis=1)

In [6]:
# -----------------------------------------------------------------
# 3. TRAIN / TEST SPLIT (stratified — critical given the imbalance)
# -----------------------------------------------------------------
X = df.drop("Class", axis=1)
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# WHY SMOTE only on training data: applying it before the split would
# leak synthetic copies of test-set fraud cases into training, inflating
# your metrics artificially. This is a very common beginner mistake —
# mentioning you avoided it is a strong interview signal.
sm = SMOTE(random_state=RANDOM_STATE)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)
print("After SMOTE:", y_train_res.value_counts().to_dict())


After SMOTE: {0: 227451, 1: 227451}


In [7]:
# -----------------------------------------------------------------
# 4. MODEL 1 — Logistic Regression (baseline, interpretable)
# -----------------------------------------------------------------
log_reg = LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)
log_reg.fit(X_train_res, y_train_res)
y_pred_lr = log_reg.predict(X_test)
y_prob_lr = log_reg.predict_proba(X_test)[:, 1]

print("\n--- Logistic Regression ---")
print(classification_report(y_test, y_pred_lr, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_lr))
print("PR-AUC:", average_precision_score(y_test, y_prob_lr))


--- Logistic Regression ---
              precision    recall  f1-score   support

           0     0.9999    0.9744    0.9869     56864
           1     0.0581    0.9184    0.1094        98

    accuracy                         0.9743     56962
   macro avg     0.5290    0.9464    0.5481     56962
weighted avg     0.9982    0.9743    0.9854     56962

ROC-AUC: 0.9698482164390798
PR-AUC: 0.7248611768135257


In [8]:
# -----------------------------------------------------------------
# 5. MODEL 2 — XGBoost (stronger, handles imbalance natively too)
# -----------------------------------------------------------------
# scale_pos_weight lets XGBoost handle imbalance without needing SMOTE;
# training both ways (with/without SMOTE) is a good ablation to mention.
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=scale_pos_weight,
    eval_metric="aucpr",
    random_state=RANDOM_STATE,
)
xgb_model.fit(X_train, y_train)  # note: raw imbalanced data + scale_pos_weight
y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

print("\n--- XGBoost ---")
print(classification_report(y_test, y_pred_xgb, digits=4))
print("ROC-AUC:", roc_auc_score(y_test, y_prob_xgb))
print("PR-AUC:", average_precision_score(y_test, y_prob_xgb))



--- XGBoost ---
              precision    recall  f1-score   support

           0     0.9997    0.9994    0.9996     56864
           1     0.7217    0.8469    0.7793        98

    accuracy                         0.9992     56962
   macro avg     0.8607    0.9232    0.8895     56962
weighted avg     0.9993    0.9992    0.9992     56962

ROC-AUC: 0.9797330616264514
PR-AUC: 0.8634284709101236


In [9]:
# -----------------------------------------------------------------
# 6. WHY PR-AUC MATTERS MORE THAN ROC-AUC HERE
# -----------------------------------------------------------------
# With 0.17% positive class, ROC-AUC can look great (>0.97) even for a
# mediocre model, because true negatives dominate. Precision-Recall AUC
# is the honest metric for extreme imbalance — say this explicitly in
# your README/interview, it shows real understanding, not just copying
# metrics off a Kaggle notebook.

precision, recall, thresholds = precision_recall_curve(y_test, y_prob_xgb)
plt.figure(figsize=(6, 5))
plt.plot(recall, precision)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve — XGBoost")
plt.savefig("outputs/pr_curve.png", bbox_inches="tight")
plt.close()


In [10]:
# -----------------------------------------------------------------
# 7. CONFUSION MATRIX AT A BUSINESS-CHOSEN THRESHOLD
# -----------------------------------------------------------------
# Default 0.5 threshold is usually wrong for fraud. Pick a threshold
# based on the cost tradeoff: missing fraud (false negative) is usually
# far costlier than a false alarm. Example: pick threshold that gives
# recall >= 0.85.
target_recall = 0.85
# recall is monotonically non-increasing as threshold rises, and
# thresholds has one fewer element than precision/recall (the final
# precision/recall pair has no associated threshold). We want the
# LAST index where recall is still >= target -> the highest threshold
# that still meets the recall bar, which maximizes precision at that
# recall level. (Using the FIRST such index, as before, picks the
# lowest threshold -> flags almost everything -> precision collapses.)
valid_idx = np.where(recall[:-1] >= target_recall)[0]
if len(valid_idx) > 0:
    idx = valid_idx[-1]
    chosen_threshold = thresholds[idx]
else:
    # no threshold achieves the target recall -> fall back to default
    chosen_threshold = 0.5
y_pred_business = (y_prob_xgb >= chosen_threshold).astype(int)
 
print(f"\nChosen threshold for recall>={target_recall}: {chosen_threshold:.4f}")
print(confusion_matrix(y_test, y_pred_business))
print(classification_report(y_test, y_pred_business, digits=4))
 


Chosen threshold for recall>=0.85: 0.3274
[[56810    54]
 [   14    84]]
              precision    recall  f1-score   support

           0     0.9998    0.9991    0.9994     56864
           1     0.6087    0.8571    0.7119        98

    accuracy                         0.9988     56962
   macro avg     0.8042    0.9281    0.8556     56962
weighted avg     0.9991    0.9988    0.9989     56962



In [11]:
# -----------------------------------------------------------------
# 8. EXPLAINABILITY WITH SHAP
# -----------------------------------------------------------------
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test.iloc[:1000])  # sample for speed

shap.summary_plot(shap_values, X_test.iloc[:1000], show=False)
plt.savefig("outputs/shap_summary.png", bbox_inches="tight")
plt.close()

In [12]:
# -----------------------------------------------------------------
# 9. SAVE ARTIFACTS FOR THE STREAMLIT APP
# -----------------------------------------------------------------
joblib.dump(xgb_model, "outputs/xgb_fraud_model.pkl")
joblib.dump(scaler, "outputs/scaler.pkl")
joblib.dump(list(X.columns), "outputs/feature_columns.pkl")
joblib.dump(chosen_threshold, "outputs/chosen_threshold.pkl")
 
print("\nDone. Model + artifacts saved to outputs/")
 


Done. Model + artifacts saved to outputs/
